In [0]:
dbutils.widgets.removeAll()

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

In [0]:
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema_origen", "bronze")
dbutils.widgets.text("esquema_destino", "silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_origen = dbutils.widgets.get("esquema_origen")
esquema_destino = dbutils.widgets.get("esquema_destino")

tabla_clientes = f"{catalogo}.{esquema_origen}.clientes"
tabla_productos = f"{catalogo}.{esquema_origen}.productos"
tabla_ordenes = f"{catalogo}.{esquema_origen}.ordenes"
tabla_detalle = f"{catalogo}.{esquema_origen}.detalle_ordenes"

tabla_destino = (
    f"{catalogo}.{esquema_destino}.ventas_detalle_limpias"
)

print(f"Destino: {tabla_destino}")

In [0]:
df_clientes_raw = spark.table(tabla_clientes)
df_productos_raw = spark.table(tabla_productos)
df_ordenes_raw = spark.table(tabla_ordenes)
df_detalle_raw = spark.table(tabla_detalle)

print(f"Clientes Bronze: {df_clientes_raw.count()}")
print(f"Productos Bronze: {df_productos_raw.count()}")
print(f"Órdenes Bronze: {df_ordenes_raw.count()}")
print(f"Detalles Bronze: {df_detalle_raw.count()}")

In [0]:
df_clientes_limpios = (
    df_clientes_raw

    .withColumn(
        "cliente_id_limpio",
        F.expr(
            "try_cast(try_cast(trim(cliente_id) as double) as bigint)"
        )
    )
    .withColumn(
        "nombre_limpio",
        F.when(
            F.col("nombre").isNull()
            | (F.trim(F.col("nombre")) == ""),
            None
        ).otherwise(
            F.initcap(F.trim(F.col("nombre")))
        )
    )
    .withColumn(
        "email_limpio",
        F.when(
            F.col("email").isNull()
            | (F.trim(F.col("email")) == ""),
            None
        ).otherwise(
            F.lower(F.trim(F.col("email")))
        )
    )
    .withColumn(
        "estado_limpio",
        F.when(
            F.col("estado").isNull()
            | (F.trim(F.col("estado")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("estado")))
        )
    )
    .withColumn(
        "fecha_registro_limpia",
        F.expr("try_cast(trim(fecha_registro) as date)")
    )
    .withColumn(
        "edad_numero",
        F.expr(
            "try_cast(try_cast(trim(edad) as double) as int)"
        )
    )
    .withColumn(
        "edad_limpia",
        F.when(
            F.col("edad_numero").between(18, 100),
            F.col("edad_numero")
        )
    )
    .withColumn(
        "genero_limpio",
        F.when(
            F.lower(F.trim(F.col("genero"))).isin(
                "m", "masculino", "hombre"
            ),
            "Masculino"
        )
        .when(
            F.lower(F.trim(F.col("genero"))).isin(
                "f", "femenino", "mujer"
            ),
            "Femenino"
        )
        .otherwise("No especificado")
    )

    .filter(F.col("cliente_id_limpio").isNotNull())

    .withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("cliente_id_limpio")
            .orderBy(F.col("ingestion_date").desc_nulls_last())
        )
    )
    .filter(F.col("_rn") == 1)

    .select(
        F.col("cliente_id_limpio").alias("cliente_id"),
        F.coalesce(
            F.col("nombre_limpio"),
            F.lit("Cliente sin nombre")
        ).alias("nombre_cliente"),
        F.col("email_limpio").alias("email"),
        F.col("estado_limpio").alias("estado_cliente"),
        F.col("fecha_registro_limpia").alias("fecha_registro"),
        F.col("edad_limpia").alias("edad"),
        F.col("genero_limpio").alias("genero")
    )
)

print(f"Clientes limpios: {df_clientes_limpios.count()}")

display(df_clientes_limpios.limit(20))

In [0]:
df_productos_limpios = (
    df_productos_raw

    # Convertir producto_id de valores como "1.0" a entero
    .withColumn(
        "producto_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(producto_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    # Limpiar columnas de texto
    .withColumn(
        "sku_limpio",
        F.when(
            F.col("sku").isNull()
            | (F.trim(F.col("sku")) == ""),
            None
        ).otherwise(
            F.upper(F.trim(F.col("sku")))
        )
    )
    .withColumn(
        "producto_limpio",
        F.when(
            F.col("producto").isNull()
            | (F.trim(F.col("producto")) == ""),
            "Producto sin nombre"
        ).otherwise(
            F.initcap(F.trim(F.col("producto")))
        )
    )
    .withColumn(
        "categoria_limpia",
        F.when(
            F.col("categoria").isNull()
            | (F.trim(F.col("categoria")) == ""),
            "Sin categoría"
        ).otherwise(
            F.initcap(F.trim(F.col("categoria")))
        )
    )
    .withColumn(
        "marca_limpia",
        F.when(
            F.col("marca").isNull()
            | (F.trim(F.col("marca")) == ""),
            "Sin marca"
        ).otherwise(
            F.initcap(F.trim(F.col("marca")))
        )
    )

    # Convertir costo y precio a números
    .withColumn(
        "costo_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(trim(costo), ',', '')
                AS DOUBLE
            )
            """
        )
    )
    .withColumn(
        "precio_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(trim(precio), ',', '')
                AS DOUBLE
            )
            """
        )
    )

    # Valores monetarios inválidos quedan como NULL
    .withColumn(
        "costo_limpio",
        F.when(
            F.col("costo_numero") > 0,
            F.col("costo_numero")
        )
    )
    .withColumn(
        "precio_limpio",
        F.when(
            F.col("precio_numero") > 0,
            F.col("precio_numero")
        )
    )

    # Normalizar activo
    .withColumn(
        "activo_limpio",
        F.when(
            F.lower(F.trim(F.col("activo"))).isin(
                "si", "sí", "s", "1", "true"
            ),
            "SI"
        )
        .when(
            F.lower(F.trim(F.col("activo"))).isin(
                "no", "n", "0", "false"
            ),
            "NO"
        )
        .otherwise("NO ESPECIFICADO")
    )

    # Convertir fecha; las fechas inválidas quedan como NULL
    .withColumn(
        "fecha_alta_limpia",
        F.expr(
            "try_cast(trim(fecha_alta) AS DATE)"
        )
    )

    # Eliminar productos sin llave válida
    .filter(
        F.col("producto_id_limpio").isNotNull()
    )

    # Mantener un solo registro por producto
    .withColumn(
        "_rn",
        F.row_number().over(
            Window
            .partitionBy("producto_id_limpio")
            .orderBy(
                F.col("ingestion_date").desc_nulls_last()
            )
        )
    )
    .filter(F.col("_rn") == 1)

    .select(
        F.col("producto_id_limpio").alias("producto_id"),
        F.col("sku_limpio").alias("sku"),
        F.col("producto_limpio").alias("producto"),
        F.col("categoria_limpia").alias("categoria"),
        F.col("marca_limpia").alias("marca"),
        F.col("costo_limpio").alias("costo"),
        F.col("precio_limpio").alias("precio"),
        F.col("activo_limpio").alias("activo"),
        F.col("fecha_alta_limpia").alias("fecha_alta")
    )
)

print(
    f"Productos Bronze: {df_productos_raw.count()}"
)

print(
    f"Productos limpios: {df_productos_limpios.count()}"
)

display(df_productos_limpios.limit(20))

In [0]:
df_ordenes_limpias = (
    df_ordenes_raw

    # Convertir IDs
    .withColumn(
        "orden_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(orden_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )
    .withColumn(
        "cliente_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(cliente_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    # Convertir fechas
    .withColumn(
        "fecha_orden_limpia",
        F.expr(
            "try_cast(trim(fecha_orden) AS DATE)"
        )
    )
    .withColumn(
        "fecha_entrega_parseada",
        F.expr(
            "try_cast(trim(fecha_entrega) AS DATE)"
        )
    )

    # Una entrega anterior a la orden se considera inválida
    .withColumn(
        "fecha_entrega_limpia",
        F.when(
            F.col("fecha_entrega_parseada")
            >= F.col("fecha_orden_limpia"),
            F.col("fecha_entrega_parseada")
        )
    )

    # Limpiar textos
    .withColumn(
        "estatus_limpio",
        F.when(
            F.col("estatus").isNull()
            | (F.trim(F.col("estatus")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("estatus")))
        )
    )
    .withColumn(
        "metodo_pago_limpio",
        F.when(
            F.col("metodo_pago").isNull()
            | (F.trim(F.col("metodo_pago")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("metodo_pago")))
        )
    )
    .withColumn(
        "canal_limpio",
        F.when(
            F.col("canal").isNull()
            | (F.trim(F.col("canal")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("canal")))
        )
    )
    .withColumn(
        "estado_envio_limpio",
        F.when(
            F.col("estado_envio").isNull()
            | (F.trim(F.col("estado_envio")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("estado_envio")))
        )
    )

    # Limpiar costo de envío
    .withColumn(
        "costo_envio_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(trim(costo_envio), ',', '')
                AS DOUBLE
            )
            """
        )
    )
    .withColumn(
        "costo_envio_limpio",
        F.when(
            F.col("costo_envio_numero").isNull()
            | (F.col("costo_envio_numero") < 0),
            F.lit(0.0)
        ).otherwise(
            F.col("costo_envio_numero")
        )
    )

    # Las llaves y la fecha de orden son obligatorias
    .filter(
        F.col("orden_id_limpio").isNotNull()
        & F.col("cliente_id_limpio").isNotNull()
        & F.col("fecha_orden_limpia").isNotNull()
    )

    # Eliminar duplicados por orden
    .withColumn(
        "_rn",
        F.row_number().over(
            Window
            .partitionBy("orden_id_limpio")
            .orderBy(
                F.col("ingestion_date").desc_nulls_last()
            )
        )
    )
    .filter(F.col("_rn") == 1)

    .select(
        F.col("orden_id_limpio").alias("orden_id"),
        F.col("cliente_id_limpio").alias("cliente_id"),
        F.col("fecha_orden_limpia").alias("fecha_orden"),
        F.col("fecha_entrega_limpia").alias("fecha_entrega"),
        F.col("estatus_limpio").alias("estatus"),
        F.col("metodo_pago_limpio").alias("metodo_pago"),
        F.col("canal_limpio").alias("canal"),
        F.col("estado_envio_limpio").alias("estado_envio"),
        F.col("costo_envio_limpio").alias("costo_envio")
    )
)

print(
    f"Órdenes Bronze: {df_ordenes_raw.count()}"
)

print(
    f"Órdenes limpias: {df_ordenes_limpias.count()}"
)

display(df_ordenes_limpias.limit(20))

In [0]:
df_detalle_limpio = (
    df_detalle_raw

    # Convertir identificadores
    .withColumn(
        "detalle_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(detalle_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )
    .withColumn(
        "orden_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(orden_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )
    .withColumn(
        "producto_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(producto_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    # Convertir cantidad
    .withColumn(
        "cantidad_numero",
        F.expr(
            "try_cast(trim(cantidad) AS DOUBLE)"
        )
    )

    # Solo aceptar cantidades positivas y enteras
    .withColumn(
        "cantidad_limpia",
        F.when(
            (F.col("cantidad_numero") > 0)
            & (
                F.col("cantidad_numero")
                == F.floor(F.col("cantidad_numero"))
            ),
            F.col("cantidad_numero").cast("int")
        )
    )

    # Convertir precio
    .withColumn(
        "precio_unitario_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(
                    trim(precio_unitario),
                    ',',
                    ''
                )
                AS DOUBLE
            )
            """
        )
    )

    # Solo aceptar precios positivos
    .withColumn(
        "precio_unitario_limpio",
        F.when(
            F.col("precio_unitario_numero") > 0,
            F.col("precio_unitario_numero")
        )
    )

    # Convertir descuento
    .withColumn(
        "descuento_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(trim(descuento), ',', '')
                AS DOUBLE
            )
            """
        )
    )

    # Descuento nulo o fuera de 0 a 1 se convierte en cero
    .withColumn(
        "descuento_limpio",
        F.when(
            F.col("descuento_numero").isNull()
            | (F.col("descuento_numero") < 0)
            | (F.col("descuento_numero") > 1),
            F.lit(0.0)
        ).otherwise(
            F.col("descuento_numero")
        )
    )

    # Conservar únicamente detalles utilizables
    .filter(
        F.col("detalle_id_limpio").isNotNull()
        & F.col("orden_id_limpio").isNotNull()
        & F.col("producto_id_limpio").isNotNull()
        & F.col("cantidad_limpia").isNotNull()
        & F.col("precio_unitario_limpio").isNotNull()
    )

    # Eliminar registros duplicados
    .withColumn(
        "_rn",
        F.row_number().over(
            Window
            .partitionBy("detalle_id_limpio")
            .orderBy(
                F.col("ingestion_date").desc_nulls_last()
            )
        )
    )
    .filter(F.col("_rn") == 1)

    # Calcular importes
    .withColumn(
        "importe_bruto",
        F.round(
            F.col("cantidad_limpia")
            * F.col("precio_unitario_limpio"),
            2
        )
    )
    .withColumn(
        "importe_descuento",
        F.round(
            F.col("importe_bruto")
            * F.col("descuento_limpio"),
            2
        )
    )
    .withColumn(
        "importe_neto",
        F.round(
            F.col("importe_bruto")
            - F.col("importe_descuento"),
            2
        )
    )

    .select(
        F.col("detalle_id_limpio").alias("detalle_id"),
        F.col("orden_id_limpio").alias("orden_id"),
        F.col("producto_id_limpio").alias("producto_id"),
        F.col("cantidad_limpia").alias("cantidad"),
        F.col("precio_unitario_limpio").alias(
            "precio_unitario"
        ),
        F.col("descuento_limpio").alias("descuento"),
        F.col("importe_bruto"),
        F.col("importe_descuento"),
        F.col("importe_neto")
    )
)

print(
    f"Detalles Bronze: {df_detalle_raw.count()}"
)

print(
    f"Detalles limpios: {df_detalle_limpio.count()}"
)

display(df_detalle_limpio.limit(20))

In [0]:
df_ventas_detalle = (
    df_detalle_limpio.alias("d")

    .join(
        df_ordenes_limpias.alias("o"),
        F.col("d.orden_id") == F.col("o.orden_id"),
        "inner"
    )

    .join(
        df_clientes_limpios.alias("c"),
        F.col("o.cliente_id") == F.col("c.cliente_id"),
        "inner"
    )

    .join(
        df_productos_limpios.alias("p"),
        F.col("d.producto_id") == F.col("p.producto_id"),
        "inner"
    )

    # Calcular días de entrega
    .withColumn(
        "dias_entrega",
        F.when(
            F.col("o.fecha_entrega").isNotNull(),
            F.datediff(
                F.col("o.fecha_entrega"),
                F.col("o.fecha_orden")
            )
        ).cast("int")
    )

    # Considerar retrasada cuando tarda más de 10 días
    .withColumn(
        "orden_retrasada",
        F.when(
            F.col("o.fecha_entrega").isNull(),
            "NO DETERMINADO"
        )
        .when(
            F.col("dias_entrega") > 10,
            "SI"
        )
        .otherwise("NO")
    )

    .select(
        F.col("o.orden_id").cast("int").alias("orden_id"),
        F.col("o.cliente_id").cast("int").alias("cliente_id"),
        F.col("c.nombre_cliente").alias("nombre_cliente"),
        F.col("c.email").alias("email"),
        F.col("c.estado_cliente").alias("estado_cliente"),
        F.col("o.fecha_orden").alias("fecha_orden"),
        F.col("o.fecha_entrega").alias("fecha_entrega"),
        F.col("o.estatus").alias("estatus"),
        F.col("o.metodo_pago").alias("metodo_pago"),
        F.col("o.canal").alias("canal"),
        F.col("o.estado_envio").alias("estado_envio"),
        F.col("p.producto_id").cast("int").alias("producto_id"),
        F.col("p.sku").alias("sku"),
        F.col("p.producto").alias("producto"),
        F.col("p.categoria").alias("categoria"),
        F.col("p.marca").alias("marca"),
        F.col("d.cantidad").cast("int").alias("cantidad"),
        F.col("d.precio_unitario")
        .cast("double")
        .alias("precio_unitario"),
        F.col("d.descuento")
        .cast("double")
        .alias("descuento"),
        F.col("d.importe_bruto")
        .cast("double")
        .alias("importe_bruto"),
        F.col("d.importe_descuento")
        .cast("double")
        .alias("importe_descuento"),
        F.col("d.importe_neto")
        .cast("double")
        .alias("importe_neto"),
        F.col("dias_entrega").cast("int").alias("dias_entrega"),
        F.col("orden_retrasada")
    )
)

print(
    f"Registros después de los JOIN: "
    f"{df_ventas_detalle.count()}"
)

display(df_ventas_detalle.limit(20))

In [0]:
df_ventas_detalle.printSchema()

In [0]:
validacion_previa = df_ventas_detalle.select(
    F.count("*").alias("registros"),

    F.sum(
        F.when(
            F.col("orden_id").isNull(),
            1
        ).otherwise(0)
    ).alias("orden_id_nulo"),

    F.sum(
        F.when(
            F.col("cliente_id").isNull(),
            1
        ).otherwise(0)
    ).alias("cliente_id_nulo"),

    F.sum(
        F.when(
            F.col("producto_id").isNull(),
            1
        ).otherwise(0)
    ).alias("producto_id_nulo"),

    F.sum(
        F.when(
            F.col("cantidad") <= 0,
            1
        ).otherwise(0)
    ).alias("cantidad_invalida"),

    F.sum(
        F.when(
            F.col("precio_unitario") <= 0,
            1
        ).otherwise(0)
    ).alias("precio_invalido"),

    F.sum(
        F.when(
            (F.col("descuento") < 0)
            | (F.col("descuento") > 1),
            1
        ).otherwise(0)
    ).alias("descuento_invalido"),

    F.sum(
        F.when(
            F.col("importe_neto") < 0,
            1
        ).otherwise(0)
    ).alias("importe_negativo")
)

display(validacion_previa)

In [0]:
tabla_destino = (
    f"{catalogo}.{esquema_destino}."
    "ventas_detalle_limpias"
)

(
    df_ventas_detalle.write
    .mode("overwrite")
    .insertInto(tabla_destino)
)

print(
    f"Tabla cargada correctamente: {tabla_destino}"
)

In [0]:
df_silver = spark.table(tabla_destino)

cantidad_dataframe = df_ventas_detalle.count()
cantidad_silver = df_silver.count()

print(
    f"Registros del DataFrame final: "
    f"{cantidad_dataframe}"
)

print(
    f"Registros cargados en Silver: "
    f"{cantidad_silver}"
)

if cantidad_dataframe != cantidad_silver:
    raise ValueError(
        "La cantidad del DataFrame no coincide con Silver."
    )

print(
    "Carga de ventas_detalle_limpias "
    "completada correctamente."
)

display(df_silver.limit(20))

In [0]:
validacion_final = df_silver.select(
    F.count("*").alias("registros"),

    F.sum(
        F.when(
            F.col("orden_id").isNull(),
            1
        ).otherwise(0)
    ).alias("orden_id_nulo"),

    F.sum(
        F.when(
            F.col("cliente_id").isNull(),
            1
        ).otherwise(0)
    ).alias("cliente_id_nulo"),

    F.sum(
        F.when(
            F.col("producto_id").isNull(),
            1
        ).otherwise(0)
    ).alias("producto_id_nulo"),

    F.sum(
        F.when(
            F.col("cantidad") <= 0,
            1
        ).otherwise(0)
    ).alias("cantidad_invalida"),

    F.sum(
        F.when(
            F.col("precio_unitario") <= 0,
            1
        ).otherwise(0)
    ).alias("precio_invalido"),

    F.sum(
        F.when(
            (F.col("descuento") < 0)
            | (F.col("descuento") > 1),
            1
        ).otherwise(0)
    ).alias("descuento_invalido"),

    F.sum(
        F.when(
            F.col("importe_neto") < 0,
            1
        ).otherwise(0)
    ).alias("importe_negativo")
)

display(validacion_final)